# Feature Engineering
Creating ML-ready features from cleaned data using fast pandas operations.

In [1]:
import pandas as pd
import numpy as np
import os
print("Ready")

Ready


In [2]:
# Load cleaned parquet
df = pd.read_parquet("../data/accidents_clean.parquet")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Shape: (500000, 44)
Columns: ['ID', 'Source', 'Severity', 'Start_Time', 'End_Time', 'Start_Lat', 'Start_Lng', 'Distance(mi)', 'Description', 'Street', 'City', 'County', 'State', 'Zipcode', 'Country', 'Timezone', 'Airport_Code', 'Weather_Timestamp', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)', 'Weather_Condition', 'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop', 'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight', 'Hour', 'DayOfWeek']


In [3]:
# Time features — use pandas vectorized operations (fast, no loops)
df['Start_Time'] = pd.to_datetime(df['Start_Time'], errors='coerce')

# Extract time components using dt accessor
df['hour_of_day'] = df['Start_Time'].dt.hour
df['day_of_week'] = df['Start_Time'].dt.dayofweek  # 0=Monday
df['month'] = df['Start_Time'].dt.month
df['year'] = df['Start_Time'].dt.year

# Derived time features
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['is_peak_hour'] = df['hour_of_day'].isin([7, 8, 9, 17, 18, 19]).astype(int)
df['is_night'] = ((df['hour_of_day'] < 6) | (df['hour_of_day'] >= 22)).astype(int)

# Season using np.select
df['season'] = np.select(
    [
        df['month'].isin([12, 1, 2]),
        df['month'].isin([3, 4, 5]),
        df['month'].isin([6, 7, 8]),
        df['month'].isin([9, 10, 11])
    ],
    ['Winter', 'Spring', 'Summer', 'Fall'],
    default='Unknown'
)

# Duration in minutes
df['duration_minutes'] = (df['End_Time'] - df['Start_Time']).dt.total_seconds() / 60
df['duration_minutes'] = df['duration_minutes'].clip(lower=0, upper=300)
df['duration_minutes'] = df['duration_minutes'].fillna(df['duration_minutes'].median())

print("Time features done")

Time features done


In [4]:
# Weather risk score — use np.select (vectorized, very fast)
weather_lower = df['Weather_Condition'].str.lower().fillna('')

df['weather_risk_score'] = np.select(
    [
        weather_lower.str.contains('tornado|hurricane|blizzard|heavy snow', na=False),
        weather_lower.str.contains('fog|smoke|snow|sleet|freezing', na=False),
        weather_lower.str.contains('rain|drizzle|thunder|hail', na=False),
        weather_lower.str.contains('cloud|overcast|mist|haze', na=False)
    ],
    [5, 4, 3, 2],
    default=1
)

df['is_adverse_weather'] = (df['weather_risk_score'] >= 3).astype(int)

print("Weather risk score distribution:")
print(df['weather_risk_score'].value_counts().sort_index())

Weather risk score distribution:
weather_risk_score
1    225098
2    237902
3     30217
4      6635
5       148
Name: count, dtype: int64


In [5]:
# Bucketing features — use pd.cut (vectorized)

# Handle nulls before cutting
if 'Visibility(mi)' in df.columns:
    df['Visibility(mi)'] = df['Visibility(mi)'].fillna(df['Visibility(mi)'].median())
    df['visibility_bucket'] = pd.cut(
        df['Visibility(mi)'],
        bins=[0, 1, 5, 999],
        labels=[0, 1, 2],
        include_lowest=True
    ).astype(int)
    df['is_low_visibility'] = (df['visibility_bucket'] == 0).astype(int)

if 'Temperature(F)' in df.columns:
    df['Temperature(F)'] = df['Temperature(F)'].fillna(df['Temperature(F)'].median())
    df['temp_bucket'] = pd.cut(
        df['Temperature(F)'],
        bins=[-999, 32, 60, 85, 999],
        labels=[0, 1, 2, 3],
        include_lowest=True
    ).astype(int)

print("Bucketing features done")

Bucketing features done


In [6]:
# Road features — encode boolean columns as int
road_cols = ['Junction', 'Traffic_Signal', 'Crossing', 'Railway', 'Station', 'Stop']
encoded_cols = []

for col in road_cols:
    if col in df.columns:
        df[col] = df[col].astype(int)
        encoded_cols.append(col)

print(f"Encoded columns: {encoded_cols}")

Encoded columns: ['Junction', 'Traffic_Signal', 'Crossing', 'Railway', 'Station', 'Stop']


In [7]:
# Target labels
df['binary_severity'] = (df['Severity'] >= 3).astype(int)

print("=== Severity distribution ===")
print(df['Severity'].value_counts().sort_index())

print("\n=== Binary severity distribution ===")
print(df['binary_severity'].value_counts())

high_pct = (df['binary_severity'].sum() / len(df)) * 100
low_pct = 100 - high_pct
print(f"\nHigh risk (Severity >= 3): {high_pct:.2f}%")
print(f"Low risk (Severity < 3): {low_pct:.2f}%")
print(f"Imbalance ratio: {low_pct/high_pct:.2f}:1")

=== Severity distribution ===
Severity
1     24061
2    289701
3    185465
4       773
Name: count, dtype: int64

=== Binary severity distribution ===
binary_severity
0    313762
1    186238
Name: count, dtype: int64

High risk (Severity >= 3): 37.25%
Low risk (Severity < 3): 62.75%
Imbalance ratio: 1.68:1


In [8]:
# Define and save feature list
ML_FEATURES = [
    'hour_of_day', 'day_of_week', 'month', 'season',
    'is_weekend', 'is_peak_hour', 'is_night',
    'weather_risk_score', 'is_adverse_weather', 'is_low_visibility',
    'visibility_bucket', 'temp_bucket', 'duration_minutes',
    'Junction', 'Traffic_Signal', 'Crossing', 'Railway'
]

# Filter to only columns that exist
available_features = [f for f in ML_FEATURES if f in df.columns]
print(f"Available features: {available_features}")

# Save full dataset
df.to_parquet("../data/accidents_features.parquet", index=False)
print(f"Full dataset saved: {df.shape}")

# Save sample for fast app loading
sample_cols = available_features + ['Severity', 'binary_severity', 'Start_Lat', 'Start_Lng']
df_sample = df[sample_cols].sample(10000, random_state=42)
df_sample.to_parquet("../data/sample_features.parquet", index=False)
print(f"Sample saved: {df_sample.shape}")

Available features: ['hour_of_day', 'day_of_week', 'month', 'season', 'is_weekend', 'is_peak_hour', 'is_night', 'weather_risk_score', 'is_adverse_weather', 'is_low_visibility', 'visibility_bucket', 'temp_bucket', 'duration_minutes', 'Junction', 'Traffic_Signal', 'Crossing', 'Railway']
Full dataset saved: (500000, 59)
Sample saved: (10000, 21)


In [9]:
# Summary
print("=== Feature Engineering Summary ===")
print(f"Features created: {available_features}")
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
high_pct = (df['binary_severity'].sum() / len(df)) * 100
low_pct = 100 - high_pct
print(f"Class imbalance: {high_pct:.2f}% high risk, {low_pct:.2f}% low risk")
print("Next: EDA notebook")

=== Feature Engineering Summary ===
Features created: ['hour_of_day', 'day_of_week', 'month', 'season', 'is_weekend', 'is_peak_hour', 'is_night', 'weather_risk_score', 'is_adverse_weather', 'is_low_visibility', 'visibility_bucket', 'temp_bucket', 'duration_minutes', 'Junction', 'Traffic_Signal', 'Crossing', 'Railway']
Dataset shape: 500000 rows, 59 columns
Class imbalance: 37.25% high risk, 62.75% low risk
Next: EDA notebook
